In [18]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [19]:
load_dotenv()  # Load environment variables from .env file

True

In [20]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [21]:
model.invoke('How far is moon from the earth?').content

"The average distance from the Earth to the Moon is approximately **384,400 kilometers (238,900 miles)**.\n\nHowever, the Moon's orbit around the Earth is not a perfect circle, but an ellipse. This means the distance varies throughout its orbit:\n\n*   **Perigee (closest point):** Approximately 363,104 km (225,623 miles)\n*   **Apogee (farthest point):** Approximately 406,696 km (252,088 miles)"

In [22]:
class LLMState(TypedDict):

    question: str
    answer: str

In [23]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = (
        "Answer in English only. "
        "Respond in plain text with no markdown formatting: "
        "no asterisks, no bullet points, no headings, and no line breaks. "
        f"Answer the following question: {question}"
    )

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state


In [24]:
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [29]:
initial_state = {'question': 'Where is Mumbai located?'}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])

Mumbai is located on the west coast of India in the state of Maharashtra.
